In [23]:
# ============================================================================
# SAFETY CHECK - Always run this first!
# ============================================================================
import os
from pathlib import Path

print("="*60)
print("📍 WORKING DIRECTORY CHECK")
print("="*60)

# Check where we are
current = Path.cwd()
print(f"\nCurrent directory: {current}")

# Verify we're in notebooks folder
if current.name != "notebooks":
    print("\n❌ ERROR: Not in notebooks/ folder!")
    print(f"   You're in: {current.name}/")
    print("\n🛑 STOP! Open this notebook from notebooks/ folder!")
    raise Exception("Wrong directory - restart from notebooks/")

# Verify data folders exist
if not Path("../data/raw").exists():
    print("\n❌ ERROR: data/raw/ folder not found!")
    raise Exception("Missing data/raw/ folder")

print("\n✅ All checks passed!")
print("✅ Safe to proceed!\n")
print("="*60)

📍 WORKING DIRECTORY CHECK

Current directory: C:\Users\kramo\OneDrive\Documents\Desktop\Projects\telecom-churn-analysis\notebooks

✅ All checks passed!
✅ Safe to proceed!



In [21]:
import os
print(os.getcwd())

C:\Users\kramo\OneDrive\Documents\Desktop\Projects\telecom-churn-analysis\notebooks


In [15]:
# DAY 2 - DATA CLEANING
# Telecom Customer Churn Analysis
# Author: Abdul Wahid Sekyere
# Date: February 6, 2026

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [4]:
print("="*70)
print("DATA CLEANING - FIXING TOTALCHARGES & VALIDATING DATA")
print("="*70)

DATA CLEANING - FIXING TOTALCHARGES & VALIDATING DATA


In [20]:
# ============================================================================
# STEP 1: LOAD THE RAW DATA
# ============================================================================
print("\n📂 Loading raw data...")
df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')
print(f"✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")


# Save original size for comparison
original_size = len(df)
print(original_size)



📂 Loading raw data...
✅ Loaded: 7,043 rows × 21 columns
7043


In [6]:
# ============================================================================
# STEP 2: INVESTIGATE TOTALCHARGES PROBLEM
# ============================================================================
print("\n" + "="*70)
print("🔍 INVESTIGATING TOTALCHARGES ISSUE")
print("="*70)

print(f"\nCurrent TotalCharges data type: {df['TotalCharges'].dtype}")

# Find the problematic rows
blank_totalcharges = df[df['TotalCharges'] == ' ']
print(f"\n📊 Found {len(blank_totalcharges)} rows with blank TotalCharges")

# Let's examine these customers
if len(blank_totalcharges) > 0:
    print("\n🔍 Examining customers with blank TotalCharges:")
    print(blank_totalcharges[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].head())

 # Check their tenure
    print(f"\nTenure of these customers:")
    print(f"  Min: {blank_totalcharges['tenure'].min()} months")
    print(f"  Max: {blank_totalcharges['tenure'].max()} months")
    print(f"  Mean: {blank_totalcharges['tenure'].mean():.1f} months")



🔍 INVESTIGATING TOTALCHARGES ISSUE

Current TotalCharges data type: object

📊 Found 11 rows with blank TotalCharges

🔍 Examining customers with blank TotalCharges:
      customerID  tenure  MonthlyCharges TotalCharges Churn
488   4472-LVYGI       0           52.55                 No
753   3115-CZMZD       0           20.25                 No
936   5709-LVOEQ       0           80.85                 No
1082  4367-NUYAO       0           25.75                 No
1340  1371-DWPAZ       0           56.05                 No

Tenure of these customers:
  Min: 0 months
  Max: 0 months
  Mean: 0.0 months


In [7]:
 # Hypothesis: These are new customers (tenure = 0)
if blank_totalcharges['tenure'].max() == 0:
        print("\n💡 INSIGHT: All blank TotalCharges are new customers (0 tenure)")
        print("   Decision: Replace blanks with 0.00 (they haven't been charged yet)")
else:
        print("\n💡 INSIGHT: Some blank TotalCharges have tenure > 0")
        print("   Decision: Replace with MonthlyCharges × tenure")


💡 INSIGHT: All blank TotalCharges are new customers (0 tenure)
   Decision: Replace blanks with 0.00 (they haven't been charged yet)


In [8]:
# ============================================================================
# STEP 3: FIX TOTALCHARGES DATA TYPE
# ============================================================================
print("\n" + "="*70)
print("🔧 FIXING TOTALCHARGES DATA TYPE")
print("="*70)

# Replace blank spaces with NaN (easier to handle)
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)

# Convert to numeric (this will convert blanks to NaN automatically)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f"\n✅ Converted TotalCharges to numeric")
print(f"   New data type: {df['TotalCharges'].dtype}")

# Check how many NaN we have now
missing_total_charges = df['TotalCharges'].isnull().sum()
print(f"   Missing values after conversion: {missing_total_charges}")



🔧 FIXING TOTALCHARGES DATA TYPE

✅ Converted TotalCharges to numeric
   New data type: float64
   Missing values after conversion: 11


In [9]:
# ============================================================================
# STEP 4: HANDLE MISSING VALUES
# ============================================================================
print("\n" + "="*70)
print("🔧 HANDLING MISSING VALUES")
print("="*70)

if missing_total_charges > 0:
    # Examine the missing values
    missing_df = df[df['TotalCharges'].isnull()]

print(f"\n📊 Customers with missing TotalCharges:")
print(missing_df[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

  # Decision logic
print("\n💭 DECISION LOGIC:")
    
  # Check if all have tenure = 0
all_zero_tenure = (missing_df['tenure'] == 0).all()

if all_zero_tenure:
        print("   All missing TotalCharges have tenure = 0")
        print("   → Filling with 0.00 (not yet charged)")
        df.loc[df['TotalCharges'].isnull(), 'TotalCharges'] = 0.0
        method_used = "Filled with 0.00"
else :
        print("   Some missing TotalCharges have tenure > 0")
        print("   → Calculating as MonthlyCharges × tenure")
        df.loc[df['TotalCharges'].isnull(), 'TotalCharges'] = \
            df.loc[df['TotalCharges'].isnull(), 'MonthlyCharges'] * \
            df.loc[df['TotalCharges'].isnull(), 'tenure']
        method_used = "Calculated from MonthlyCharges × tenure"

print(f"\n✅ Fixed {missing_total_charges} missing values")
print(f"   Method: {method_used}")
        


🔧 HANDLING MISSING VALUES

📊 Customers with missing TotalCharges:
      customerID  tenure  MonthlyCharges  TotalCharges Churn
488   4472-LVYGI       0           52.55           NaN    No
753   3115-CZMZD       0           20.25           NaN    No
936   5709-LVOEQ       0           80.85           NaN    No
1082  4367-NUYAO       0           25.75           NaN    No
1340  1371-DWPAZ       0           56.05           NaN    No
3331  7644-OMVMY       0           19.85           NaN    No
3826  3213-VVOLG       0           25.35           NaN    No
4380  2520-SGTTA       0           20.00           NaN    No
5218  2923-ARZLG       0           19.70           NaN    No
6670  4075-WKNIU       0           73.35           NaN    No
6754  2775-SEFEE       0           61.90           NaN    No

💭 DECISION LOGIC:
   All missing TotalCharges have tenure = 0
   → Filling with 0.00 (not yet charged)

✅ Fixed 11 missing values
   Method: Filled with 0.00


In [10]:
# ============================================================================
# STEP 5: VERIFY THE FIX
# ============================================================================
print("\n" + "="*70)
print("✅ VERIFICATION - CHECKING IF FIX WORKED")
print("="*70)

# Check data type
print(f"\nTotalCharges data type: {df['TotalCharges'].dtype}")
assert df['TotalCharges'].dtype in ['float64', 'int64'], "TotalCharges should be numeric!"
print("✅ Data type is numeric")

# Check for missing values
missing_check = df['TotalCharges'].isnull().sum()
print(f"\nMissing values in TotalCharges: {missing_check}")
assert missing_check == 0, "Should have no missing values!"
print("✅ No missing values")

# Check for negative values (shouldn't exist)
negative_values = (df['TotalCharges'] < 0).sum()
print(f"\nNegative values in TotalCharges: {negative_values}")
assert negative_values == 0, "Shouldn't have negative charges!"
print("✅ No negative values")

# Sample the data
print("\n📊 Sample of cleaned TotalCharges:")
print(df[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head(10))


✅ VERIFICATION - CHECKING IF FIX WORKED

TotalCharges data type: float64
✅ Data type is numeric

Missing values in TotalCharges: 0
✅ No missing values

Negative values in TotalCharges: 0
✅ No negative values

📊 Sample of cleaned TotalCharges:
   customerID  tenure  MonthlyCharges  TotalCharges
0  7590-VHVEG       1           29.85         29.85
1  5575-GNVDE      34           56.95       1889.50
2  3668-QPYBK       2           53.85        108.15
3  7795-CFOCW      45           42.30       1840.75
4  9237-HQITU       2           70.70        151.65
5  9305-CDSKC       8           99.65        820.50
6  1452-KIOVK      22           89.10       1949.40
7  6713-OKOMC      10           29.75        301.90
8  7892-POOKP      28          104.80       3046.05
9  6388-TABGU      62           56.15       3487.95


In [11]:
# ============================================================================
# STEP 6: CHECK FOR OTHER DATA QUALITY ISSUES
# ============================================================================
print("\n" + "="*70)
print("🔍 CHECKING FOR OTHER DATA QUALITY ISSUES")
print("="*70)

# Check all columns for missing values
print("\n📊 Missing values across all columns:")
missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0]

if len(missing_summary) > 0:
    print(missing_summary)
    print("\n⚠️  Found missing values in other columns!")
else:
    print("✅ No missing values in any column!")

# Check for duplicates
duplicates = df.duplicated(subset='customerID').sum()
print(f"\n📊 Duplicate customer IDs: {duplicates}")
if duplicates > 0:
    print("⚠️  Found duplicate customers - need to investigate!")
    print(df[df.duplicated(subset='customerID', keep=False)])
else:
    print("✅ No duplicate customers")

# Check data ranges
print("\n📊 Data range validation:")

# Tenure should be 0-72 months (reasonable for telecom)
print(f"Tenure: {df['tenure'].min()} to {df['tenure'].max()} months ✅")

# MonthlyCharges should be positive
print(f"MonthlyCharges: ${df['MonthlyCharges'].min():.2f} to ${df['MonthlyCharges'].max():.2f} ✅")

# TotalCharges should be >= 0
print(f"TotalCharges: ${df['TotalCharges'].min():.2f} to ${df['TotalCharges'].max():.2f} ✅")

# SeniorCitizen should be 0 or 1
print(f"SeniorCitizen unique values: {df['SeniorCitizen'].unique()} ✅")


🔍 CHECKING FOR OTHER DATA QUALITY ISSUES

📊 Missing values across all columns:
✅ No missing values in any column!

📊 Duplicate customer IDs: 0
✅ No duplicate customers

📊 Data range validation:
Tenure: 0 to 72 months ✅
MonthlyCharges: $18.25 to $118.75 ✅
TotalCharges: $0.00 to $8684.80 ✅
SeniorCitizen unique values: [0 1] ✅


In [12]:
# ============================================================================
# STEP 7: CREATE BINARY TARGET VARIABLE
# ============================================================================
print("\n" + "="*70)
print("🔧 CREATING BINARY TARGET VARIABLE")
print("="*70)

# Convert Churn to numeric (0/1) for modeling
df['Churn_Binary'] = df['Churn'].map({'No': 0, 'Yes': 1})

print("\nChurn encoding:")
print(df[['Churn', 'Churn_Binary']].value_counts())

# Verify
print(f"\n✅ Created Churn_Binary column")
print(f"   0 (Not Churned): {(df['Churn_Binary'] == 0).sum():,}")
print(f"   1 (Churned): {(df['Churn_Binary'] == 1).sum():,}")



🔧 CREATING BINARY TARGET VARIABLE

Churn encoding:
Churn  Churn_Binary
No     0               5174
Yes    1               1869
Name: count, dtype: int64

✅ Created Churn_Binary column
   0 (Not Churned): 5,174
   1 (Churned): 1,869


In [13]:
# ============================================================================
# STEP 8: FINAL DATA SUMMARY
# ============================================================================
print("\n" + "="*70)
print("📊 CLEANED DATA SUMMARY")
print("="*70)

print(f"\nOriginal dataset: {original_size:,} rows")
print(f"Cleaned dataset: {len(df):,} rows")
print(f"Rows removed: {original_size - len(df)}")

print(f"\nData types after cleaning:")
print(df.dtypes)

print("\n✅ KEY ACCOMPLISHMENTS:")
print("  ✓ Fixed TotalCharges data type (object → float64)")
print("  ✓ Handled 11 missing values in TotalCharges")
print("  ✓ Verified no other missing values")
print("  ✓ Checked for duplicates (none found)")
print("  ✓ Validated data ranges")
print("  ✓ Created binary target variable for modeling")


📊 CLEANED DATA SUMMARY

Original dataset: 7,043 rows
Cleaned dataset: 7,043 rows
Rows removed: 0

Data types after cleaning:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
Churn_Binary          int64
dtype: object

✅ KEY ACCOMPLISHMENTS:
  ✓ Fixed TotalCharges data type (object → float64)
  ✓ Handled 11 missing values in TotalCharges
  ✓ Verified no other missing values
  ✓ Checked for duplicates (none found)
  ✓ Validated data ranges
  ✓ Created binary 

In [15]:
# ============================================================================
# STEP 9: SAVE CLEANED DATA
# ============================================================================
print("\n" + "="*70)
print("💾 SAVING CLEANED DATA")
print("="*70)

# Create processed folder if it doesn't exist
import os
os.makedirs('../data/processed', exist_ok=True)

# Save cleaned dataset
output_file = '../data/processed/telecom_churn_cleaned.csv'
df.to_csv(output_file, index=False)

print(f"\n✅ Saved cleaned data to: {output_file}")
print(f"   Size: {len(df):,} rows × {len(df.columns)} columns")



💾 SAVING CLEANED DATA

✅ Saved cleaned data to: ../data/processed/telecom_churn_cleaned.csv
   Size: 7,043 rows × 22 columns


In [16]:
# ============================================================================
# STEP 10: DOCUMENT CLEANING DECISIONS
# ============================================================================
print("\n" + "="*70)
print("📝 CLEANING DECISIONS DOCUMENTED")
print("="*70)

cleaning_log = f"""
DATA CLEANING LOG - {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
================================================================
1. ISSUE: TotalCharges stored as object (string) instead of numeric
   SOLUTION: Converted to float64 using pd.to_numeric()
   RESULT: ✅ Successfully converted

2. ISSUE: 11 rows with blank TotalCharges (' ')
   INVESTIGATION: All had tenure = 0 months (new customers)
   SOLUTION: Replaced blanks with 0.00 (not yet charged)
   RESULT: ✅ Fixed all missing values

3. ISSUE: Target variable in text format ('Yes'/'No')
   SOLUTION: Created Churn_Binary (0/1) for modeling
   RESULT: ✅ Binary variable created

4. VERIFICATION CHECKS:
   ✅ No duplicate customer IDs
   ✅ No remaining missing values
   ✅ All numeric columns have valid ranges
   ✅ All data types correct

5. FINAL DATASET:
   Rows: {len(df):,}
   Columns: {len(df.columns)}
   Churn Rate: {df['Churn_Binary'].mean()*100:.2f}%
   
================================================================
Data is now ready for Exploratory Data Analysis (EDA)!
"""

print(cleaning_log)

# Save log to file (with UTF-8 encoding to handle emojis)
with open('../data/processed/cleaning_log.txt', 'w', encoding='utf-8') as f:
    f.write(cleaning_log)


print("\n✅ Cleaning log saved to: data/processed/cleaning_log.txt")




📝 CLEANING DECISIONS DOCUMENTED

DATA CLEANING LOG - 2026-02-10 22:18:24
1. ISSUE: TotalCharges stored as object (string) instead of numeric
   SOLUTION: Converted to float64 using pd.to_numeric()
   RESULT: ✅ Successfully converted

2. ISSUE: 11 rows with blank TotalCharges (' ')
   INVESTIGATION: All had tenure = 0 months (new customers)
   SOLUTION: Replaced blanks with 0.00 (not yet charged)
   RESULT: ✅ Fixed all missing values

3. ISSUE: Target variable in text format ('Yes'/'No')
   SOLUTION: Created Churn_Binary (0/1) for modeling
   RESULT: ✅ Binary variable created

4. VERIFICATION CHECKS:
   ✅ No duplicate customer IDs
   ✅ No remaining missing values
   ✅ All numeric columns have valid ranges
   ✅ All data types correct

5. FINAL DATASET:
   Rows: 7,043
   Columns: 22
   Churn Rate: 26.54%
   
Data is now ready for Exploratory Data Analysis (EDA)!


✅ Cleaning log saved to: data/processed/cleaning_log.txt


In [18]:
import os
print(os.getcwd())

C:\Users\kramo\OneDrive\Documents\Desktop\Projects\telecom-churn-analysis\notebooks
